# 02 — LexRank (estrattivo)

**LexRank** (Erkan & Radev, 2004) è, come TextRank, un metodo estrattivo basato su grafi e
PageRank; la differenza sta nella rappresentazione delle frasi: qui la similarità è la cosine
similarity tra vettori **TF-IDF** (niente reti neurali, niente download di modelli), il che lo
rende molto più veloce di TextRank su CPU. Il riassunto è composto dalle `N_SENTENCES` frasi con
punteggio più alto, nella loro forma originale.

Come il notebook 01, opera su due ambiti (`SCOPE`):
- **`sample`** — il campione condiviso di [00_prepara_campione.ipynb](00_prepara_campione.ipynb),
  per il confronto con i metodi abstractive;
- **`full`** — l'intero dataset pulito (`data/tab/complete.tab`, 56.101 esempi, in streaming),
  per il confronto tra estrattivi. Essendo TF-IDF puro, la corsa completa è fattibile anche su
  CPU (ore, non giorni).

Riassunti salvati incrementalmente in `results/summaries/lexrank_{scope}.tsv` con **ripresa**
automatica dopo un'interruzione; metriche ricalcolabili dai soli file salvati.

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [2]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO      = 'lexrank'
SCOPE       = 'sample'   # 'sample' = campione condiviso | 'full' = intero complete.tab
N_SAMPLES   = 100        # deve combaciare con il campione creato dal notebook 00
SEED        = 42
LIMIT       = None       # es. 3 per uno smoke test rapido; None = tutti
N_SENTENCES = 11         # frasi estratte per riassunto (mediana del corpus: 11 frasi/riassunto)
STOP_WORDS  = ['en']     # pre-processing per il ranking (l'output resta il testo originale)

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

print(f'Ambito (SCOPE) : {SCOPE}')
print(f'Output         : {OUT_PATH}')

Ambito (SCOPE) : sample
Output         : C:\Users\agira\source\repos\multi-news-ai4stem-polito-master\results\summaries\lexrank_sample.tsv


## Generazione dei riassunti

Per ogni documento: separatore `` ||||| `` sostituito con un newline, istanza `psr.summarization`
(il pre-processing serve solo al ranking), rank con `summ_lex_rank` (TF-IDF + PageRank) ed
estrazione delle prime `N_SENTENCES` frasi con `show_summary`. Nessun modello da caricare.

In [3]:
from pyAutoSummarizer.base import psr

def genera(documento):
    s = psr.summarization(documento, stop_words=STOP_WORDS)
    rank = s.summ_lex_rank()
    return s.show_summary(rank, n=N_SENTENCES)

esempi = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
          else su.itera_complete_tab(P['complete_tab']))
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='LexRank ')
scrittore.chiudi()

C:\Users\agira\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LexRank [1] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank [2] media 0.4 s/esempio (saltati 6 gia' fatti)


LexRank [3] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank [10] media 0.3 s/esempio (saltati 6 gia' fatti)


...

LexRank [60] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank [70] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank [80] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank [90] media 0.3 s/esempio (saltati 6 gia' fatti)


LexRank Completato: 94 nuovi, 6 saltati, 1 errori, 25 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/lexrank_{scope}_per_example.csv` e `…_aggregate.json`
(medie complessive e per split).

In [4]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
config = {'n_sentences': N_SENTENCES, 'stop_words': STOP_WORDS,
          'libreria': 'pyAutoSummarizer 1.2.0'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))

Metriche per-esempio : C:\Users\agira\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_sample_per_example.csv (99 righe)
Metriche aggregate   : C:\Users\agira\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_sample_aggregate.json
{
  "rouge1_f1": 0.3747382976055879,
  "rouge1_p": 0.32177741277968025,
  "rouge1_r": 0.480013280862619,
  "rouge2_f1": 0.14072258013486186,
  "rouge2_p": 0.11444316613609484,
  "rouge2_r": 0.19732675974538255,
  "rougeL_f1": 0.17335243665301764,
  "rougeL_p": 0.1355579279420092,
  "rougeL_r": 0.2620149579200482,
  "bleu": 0.08862086399199129,
  "meteor": 0.5105332585274158,
  "parole_generate": 436.1010101010101
}


## Ispezione qualitativa

In [5]:
riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
su.mostra_esempi(riferimenti, riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : The Jewish campaign treasurer for a Republican candidate in Connecticut is defending a campaign mailer that depicts their Democratic Party opponent in what is being described as the “worst kind of anti-Semitism.” 
 
 “The full-color, double-sided mailer shows an altered image of Democrat Matt Lesser, who is Jewish, with large, beady eyes, holding fistfuls of hundred-dollar bills — Gabe Rosenberg (@Gaber205) October 30, 2018
A mailer sent out by Republican state Senate candidate Ed Charamut in Co
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was 